# 🧬 Evo2 Tools Example

This notebook demonstrates how to generate and score DNA sequences using **Evo2**.

## 📖 What is Evo2?

[Evo2](https://www.science.org/doi/10.1126/science.ado9336) is a large autoregressive DNA language model trained on billions of nucleotide sequences, built on the Vortex architecture for efficient long-context inference.

### Key Features:
- **Generation** -- Autoregressive DNA sequence generation from prompts
- **Scoring** -- Autoregressive likelihood scoring to assess sequence naturalness
- **GPU-ready** -- Optimized for CUDA execution with efficient batching
- **Flexible sampling** -- Temperature, top-k, and top-p controls for diverse generation

## 📥 Imports

## 📦 Shared Data Types

### `SequenceScores`
Scoring metrics for a single sequence, returned by the scoring tool. Metrics can be accessed via dict-style (`score.metrics["perplexity"]`) or attribute-style (`score.perplexity`).

| Field | Type | Description |
|-------|------|-------------|
| `metrics` | `Dict[str, float]` | Scoring metrics: `log_likelihood`, `avg_log_likelihood`, `perplexity` |
| `logits` | `Optional[List[List[float]]]` | Per-position logits (seq_len, vocab_size). Only present if `return_logits=True` |
| `vocab` | `Optional[List[str]]` | Token ordering for logits columns |

In [1]:
from bio_programming_tools import (
    Evo2SampleConfig,
    Evo2SampleInput,
    Evo2ScoringConfig,
    Evo2ScoringInput,
    run_evo2_sample,
    run_evo2_score,
)


## 🧪 1. Generate DNA Sequences

Generate DNA sequences from a prompt using autoregressive sampling.

Evo2 extends a given DNA prompt by predicting the most likely nucleotides one at a time, using its learned distribution over billions of natural sequences. The sampling parameters control the diversity and quality of the generated output.

### API Reference

**`Evo2SampleInput`**

| Field | Type | Description |
|-------|------|-------------|
| `prompts` | `List[str]` | Prompt sequences for DNA generation |

**`Evo2SampleConfig`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_checkpoint` | `str` | `"evo2_7b"` | Evo2 model checkpoint to use |
| `num_tokens` | `int` | `32` | Number of tokens to generate (does not include prompt) |
| `temperature` | `float` | `1.0` | Scales the randomness of sampling |
| `top_k` | `int` | `4` | Limits sampling to the top-k most probable tokens |
| `top_p` | `float` | `1.0` | Nucleus sampling parameter (cumulative probability cutoff) |
| `prepend_prompt` | `bool` | `True` | Whether to prepend the prompt to the generated sequence |
| `cached_generation` | `bool` | `True` | Whether to use vortex KV caching for generation |
| `stop_at_eos` | `bool` | `True` | Whether to stop at end-of-sequence token |
| `batch_size` | `Optional[int]` | `None` | Max number of samples on the GPU at once |
| `return_logits` | `bool` | `False` | Whether to include per-position logits in the output |

**`Evo2SampleOutput`**

| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Generated DNA sequences |

In [2]:
inputs = Evo2SampleInput(prompts=["ATGCGTAAA"])

# Configure generation parameters
config = Evo2SampleConfig(
    num_tokens=200,
    temperature=0.8,
    top_k=4,
)

sample_result = run_evo2_sample(inputs, config)

# Display the generated sequence
prompt = inputs.prompts[0]
generated = sample_result.sequences[0]
print(f"Prompt:       {prompt}")
print(f"Generated:    {generated[:80]}..." if len(generated) > 80 else f"Generated:    {generated}")
print(f"Total length: {len(generated)} nucleotides")


[02/06/26 00:26:56] WARNING  transformer_engine.pytorch.attention.dot_product_attention.utils -      ]8;id=712116;file:///home/daniel.guo/miniconda/envs/bio-programming/lib/python3.12/site-packages/transformer_engine/pytorch/attention/dot_product_attention/backends.py\backends.py]8;;\:]8;id=401315;file:///home/daniel.guo/miniconda/envs/bio-programming/lib/python3.12/site-packages/transformer_engine/pytorch/attention/dot_product_attention/backends.py#98\98]8;;\
                             WARNING - Supported flash-attn versions are >= 2.1.1, <= 2.7.4.post1.                 
                             Found flash-attn 2.8.0.post2.                                                         

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Found complete file in repo: evo2_7b.pt


100%|██████████| 32/32 [00:00<00:00, 174.57it/s]


Extra keys in state_dict: {'blocks.10.mixer.attn._extra_state', 'blocks.6.mixer.mixer.filter.t', 'blocks.2.mixer.mixer.filter.t', 'blocks.30.mixer.mixer.filter.t', 'blocks.31.mixer.attn._extra_state', 'unembed.weight', 'blocks.17.mixer.dense._extra_state', 'blocks.27.mixer.mixer.filter.t', 'blocks.3.mixer.attn._extra_state', 'blocks.17.mixer.attn._extra_state', 'blocks.31.mixer.dense._extra_state', 'blocks.3.mixer.dense._extra_state', 'blocks.24.mixer.attn._extra_state', 'blocks.10.mixer.dense._extra_state', 'blocks.20.mixer.mixer.filter.t', 'blocks.9.mixer.mixer.filter.t', 'blocks.24.mixer.dense._extra_state', 'blocks.16.mixer.mixer.filter.t', 'blocks.23.mixer.mixer.filter.t', 'blocks.13.mixer.mixer.filter.t'}
Initializing inference params with max_seqlen=209


[02/06/26 00:27:07] WARNING  bio_programming_tools.tools.tool_registry - WARNING - Tool evo2-sample: ]8;id=594944;file:///home/daniel.guo/bio-programming/bio_programming/tools/tool_registry.py\tool_registry.py]8;;\:]8;id=804424;file:///home/daniel.guo/bio-programming/bio_programming/tools/tool_registry.py#184\184]8;;\
                             Casting complex values to real discards the imaginary part                            
                             (Triggered internally at                                                              
                             /home/conda/feedstock_root/build_artifacts/libtorch_1753846581042                     
                             /work/aten/src/ATen/native/Copy.cpp:307.)                                             

Prompt:       ATGCGTAAA
Generated:    ATGCGTAAATATCAATCGATATAAATTTTTTGAAATTTTTTTGTCAATTTTAAAAATAGAACATATAAAAATTTTAAAAT...
Total length: 209 nucleotides


## 📏 2. Score Sequences

Compute autoregressive likelihood metrics for DNA sequences.

Evo2 scores sequences by computing the autoregressive log-likelihood at each position. The resulting perplexity measures how "natural" a DNA sequence looks to the model. Lower values indicate sequences that are more consistent with the patterns learned from natural genomes. This is useful for ranking generated sequences or evaluating sequence quality.

### API Reference

**`Evo2ScoringInput`**

| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | DNA sequences to score |

**`Evo2ScoringConfig`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_checkpoint` | `str` | `"evo2_7b"` | Evo2 model checkpoint to use |
| `batch_size` | `Optional[int]` | `None` | Max number of samples on the GPU at once |
| `return_logits` | `bool` | `False` | Whether to include per-position logits in the output |

**`Evo2ScoringOutput`**

| Field | Type | Description |
|-------|------|-------------|
| `scores` | `List[SequenceScores]` | List of scoring outputs, one per input sequence |

In [3]:
score_inputs = Evo2ScoringInput(sequences=["ATGCGTAAA", "ATGCGTAAATTT"])

# Configure scoring parameters
score_config = Evo2ScoringConfig(batch_size=2, return_logits=False)

score_result = run_evo2_score(score_inputs, score_config)
print(f"Sequence 1: {score_inputs.sequences[0]}")
print(f"    Perplexity: {score_result.scores[0].metrics['perplexity']:.3f}")
print(f"Sequence 2: {score_inputs.sequences[1]}")
print(f"    Perplexity: {score_result.scores[1].metrics['perplexity']:.3f}")


Sequence 1: ATGCGTAAA
    Perplexity: 3.943
Sequence 2: ATGCGTAAATTT
    Perplexity: 3.815


## 💾 3. Export Results

Save the results to files for downstream analysis.

### Supported formats:
- **JSON** -- Structured data with metadata and all scoring information
- **FASTA** -- Simple sequence format for compatibility with other bioinformatics tools

The exported results can be used for:
- Downstream sequence analysis and annotation (generated sequences)
- Ranking and filtering candidate sequences (scoring metrics)
- Input to other bioinformatics pipelines

In [ ]:
# Export generated sequences to FASTA format
sample_result.export("./example_output/evo2_sequences", file_format="fasta")
print("Generated sequences exported to ./example_output/evo2_sequences")

# Export scores to JSON format
score_result.export("./example_output/evo2_scores", file_format="json")
print("Scores exported to ./example_output/evo2_scores")